In [1]:
import argparse
import math
import sqlite3
import yaml
import os
import sys
import json
from multiprocessing import Pool

import pandas as pd

import numpy as np
from tqdm import tqdm

In [2]:
from mol_utils import MolPreprocessor

## SQlite 条件
- ``standard_units``: %
- ``standard_value``: -10-110。範囲外のデータは信用しない
- ``relation``: "=", ">", "<", "<=", ">="。これ以外はレコード数がほとんど存在しない
- ``assay_type``: "B"。結合アッセイにしぼる
- ``target_type``: "SINGLE PROTEIN"。単一タンパク質にしぼる
- ``standard_value``: 欠損値は除去

In [3]:
con = sqlite3.connect(f"../input/chembl_34/chembl_34_sqlite/chembl_34.db")
df = pd.read_sql('SELECT activity_id, act.assay_id AS assay_id, t.chembl_id AS target_chembl_id, t.pref_name AS target_name, t.organism AS target_organism, cs.accession AS target_accession, cs.sequence AS target_sequence, '\
                'md.chembl_id as compound_chembl_id, cs.canonical_smiles as smiles, standard_type, relation, standard_value, standard_units, pchembl_value, data_validity_comment, a.description as description, '\
                'protein_classification.protein_class_desc AS protein_class_desc '\
                'FROM activities act '\
                'JOIN molecule_dictionary md ON act.molregno = md.molregno '\
                'JOIN compound_structures cs ON md.molregno = cs.molregno '\
                'JOIN assays a ON act.assay_id = a.assay_id '\
                'JOIN target_dictionary t ON a.tid = t.tid '\
                'JOIN target_components tc ON t.tid = tc.tid '\
                'JOIN component_sequences cs ON tc.component_id = cs.component_id '\
                'JOIN component_class cc ON tc.component_id = cc.component_id '\
                'JOIN protein_classification ON cc.protein_class_id = protein_classification.protein_class_id '\
                'WHERE standard_units = "%" '\
                'AND relation IN ("=", ">", "<", "<=", ">=") '\
                'AND standard_value BETWEEN -10 AND 110 '\
                'AND a.assay_type = "B" '\
                'AND t.target_type = "SINGLE PROTEIN" '\
                'AND act.standard_value IS NOT NULL' \
                , con)
con.close()

In [4]:
len(df)

655954

``data_validity_comment``は、null (問題ない)か、"Manually validated"を抽出

In [5]:
df1 = df[df["data_validity_comment"].isnull()] # null = no problem
df2 = df[df["data_validity_comment"]=="Manually validated"]
df = pd.concat([df1, df2], axis=0).sort_index().drop("data_validity_comment", axis=1).reset_index(drop=True)
len(df)

655953

## 実験条件における濃度の抽出

In [6]:
import re

def extract_regex_group(desc, pattern, group_num, ignore_case: bool = False):
    """descriptionから正規表現で指定したグループを抽出する."""
    flags = re.I if ignore_case else 0
    match = re.search(pattern, desc, flags)
    if match:
        return match.group(group_num)
    return None

pattern = r'(\d+(\.\d+)?)\s*([mun]M)'
assay_conc_full = [extract_regex_group(x, pattern, 0, False) if extract_regex_group(x, pattern, 0, False) else None for x in df["description"]]
df["assay_conc_full"] = assay_conc_full

assay_conc = []
assay_conc_units = []
for a_conc_full in df["assay_conc_full"]:
    if not a_conc_full:
        assay_conc.append(None)
        assay_conc_units.append(None)
        continue
    match = re.search(r'(\d+(\.\d+)?)\s*([mun]M)', a_conc_full)
    if match:
        number = match.group(1)
        unit = match.group(3)
        assay_conc.append(number)
        assay_conc_units.append(unit)

df["assay_conc"] = assay_conc
df["assay_conc_units"] = assay_conc_units
df["assay_conc_full"].isnull().sum()

90935

In [7]:
df["assay_conc_units"].value_counts()

uM    529512
nM     26308
mM      9198
Name: assay_conc_units, dtype: int64

In [8]:
#濃度情報のないデータを削除
df_conc_not_null = df[df["assay_conc"].notnull()].reset_index(drop=True)
df_conc_not_null

,activity_id,assay_id,target_chembl_id,target_name,target_organism,target_accession,target_sequence,compound_chembl_id,smiles,standard_type,relation,standard_value,standard_units,pchembl_value,description,protein_class_desc,assay_conc_full,assay_conc,assay_conc_units
0,31911,50861,CHEMBL1978,Cytochrome P450 19A1,Homo sapiens,P11511,MVLEMLNPIHYNITSIVPEAMPAATMPVLLLTGLFLLVWNYEGTSS...,CHEMBL132530,C[C@]12CCC(=O)C(O)=C1CC[C@@H]1[C@@H]2CC[C@]2(C...,Inhibition,=,50.1,%,None,"Inhibition of 0.25 uM [1-3H]-androst-4-ene-3,1...",enzyme cytochrome p450 cyp_19 cyp_19a cyp_...,0.25 uM,0.25,uM
1,31912,50862,CHEMBL1978,Cytochrome P450 19A1,Homo sapiens,P11511,MVLEMLNPIHYNITSIVPEAMPAATMPVLLLTGLFLLVWNYEGTSS...,CHEMBL132530,C[C@]12CCC(=O)C(O)=C1CC[C@@H]1[C@@H]2CC[C@]2(C...,Inhibition,=,81.6,%,None,"Inhibition of 1.25 uM [1-3H]-androst-4-ene-3,1...",enzyme cytochrome p450 cyp_19 cyp_19a cyp_...,1.25 uM,1.25,uM
2,32535,2700,CHEMBL322,Serotonin 2a (5-HT2a) receptor,Rattus norvegicus,P14842,MEILCEDNISLSSIPNSLMQLGDGPRLYHNDFNSRDANTSEASNWT...,CHEMBL6710,C[C@@H](N)Cc1c2c(c(Br)c3c1OCC3)OCC2.Cl,Max stimulation,=,80.0,%,None,Percent maximal 5-HT stimulation at cloned rat...,membrane receptor 7tm1 smallmol monoamine r...,100 uM,100,uM
3,32536,88625,CHEMBL3471,Human immunodeficiency virus type 1 integrase,Human immunodeficiency virus 1,Q7ZJM1,FLDGIDKAQDEHEKYHSNWRAMASDFNLPPVVAKEIVASCDKCQLK...,CHEMBL63875,N=C(N)NCCC[C@H](NC(N)=O)C(=O)O,Inhibition,<,50.0,%,None,Inhibition against HIV-1 integrase at 1 mM,enzyme hydrolase,1 mM,1,mM
4,32537,88626,CHEMBL3471,Human immunodeficiency virus type 1 integrase,Human immunodeficiency virus 1,Q7ZJM1,FLDGIDKAQDEHEKYHSNWRAMASDFNLPPVVAKEIVASCDKCQLK...,CHEMBL63875,N=C(N)NCCC[C@H](NC(N)=O)C(=O)O,Inhibition,<,50.0,%,None,Inhibition against HIV-1 integrase at 200 uM,enzyme hydrolase,200 uM,200,uM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
565013,25111824,2304392,CHEMBL5291573,HTH-type transcriptional regulator LuxR,Vibrio harveyi,P21308,MDSIAKRPRTRLSPLKRKQQLMEIALEVFARRGIGRGGHADIAEIA...,CHEMBL2299418,CN(C)c1ccc(/C=C/C=O)cc1,Inhibition,=,17.0,%,None,Inhibition of LuxR quorum sensing system in Vi...,unclassified,100 uM,100,uM
565014,25111825,2304392,CHEMBL5291573,HTH-type transcriptional regulator LuxR,Vibrio harveyi,P21308,MDSIAKRPRTRLSPLKRKQQLMEIALEVFARRGIGRGGHADIAEIA...,CHEMBL5289639,O=C/C=C/c1ccc([N+](=O)[O-])cc1,Inhibition,=,33.0,%,None,Inhibition of LuxR quorum sensing system in Vi...,unclassified,100 uM,100,uM
565015,25112624,2304634,CHEMBL4599,Tyrosine kinase non-receptor protein 2,Homo sapiens,Q07912,MQPEEGTGWLLELLSEVQLQQYFLRLRDDLNVTRLSHFEYVKNEDL...,CHEMBL428690,CN1CC[C@H](c2c(O)cc(O)c3c(=O)cc(-c4ccccc4Cl)oc...,Inhibition,=,74.0,%,None,Inhibition of ACK1 (unknown origin) at 10 uM r...,enzyme kinase protein kinase tk ack,10 uM,10,uM
565016,25112625,2304634,CHEMBL4599,Tyrosine kinase non-receptor protein 2,Homo sapiens,Q07912,MQPEEGTGWLLELLSEVQLQQYFLRLRDDLNVTRLSHFEYVKNEDL...,CHEMBL535,CCN(CC)CCNC(=O)c1c(C)[nH]c(/C=C2\C(=O)Nc3ccc(F...,Inhibition,=,33.0,%,None,Inhibition of ACK1 (unknown origin) at 10 uM r...,enzyme kinase protein kinase tk ack,10 uM,10,uM


In [9]:
# 単位をuMに統一
def convert_units(value, unit):
    if unit == "uM":
        return value
    elif unit == "nM":
        return value / 1000
    elif unit == "mM":
        return value * 1000

In [10]:
# 濃度情報をuMに統一
for i in range(len(df_conc_not_null)):
    value = df_conc_not_null.loc[i, "assay_conc"]
    unit = df_conc_not_null.loc[i, "assay_conc_units"]
    df_conc_not_null.at[i, "assay_conc_uM"] = convert_units(float(value), unit)

In [11]:
# 二値分類ラベルの付与をする関数
def binalize_activity(row, threshold):
    if row["relation"] == "=":
        if (row["assay_conc_uM"] <= threshold) & (row["standard_value"] >= 50):
            return 1
        elif (row["assay_conc_uM"] > threshold) & (row["standard_value"] < 50):
            return 0
        else:
            return None
    
    elif row["relation"] == ">" or row["relation"] == ">=":
        if (row["assay_conc_uM"] > threshold) & (row["standard_value"] < 50):
            return 0
        else:
            return None
    
    elif row["relation"] == "<" or row["relation"] == "<=":
        if (row["assay_conc_uM"] < threshold) & (row["standard_value"] >= 50):
            return 1
        else:
            return None

In [12]:
# 二値ラベルの付与
df_conc_not_null["t_100n"] = df_conc_not_null.apply(lambda x: binalize_activity(x, 0.1), axis=1)
df_conc_not_null["t_1u"] = df_conc_not_null.apply(lambda x: binalize_activity(x, 1), axis=1)
df_conc_not_null["t_10u"] = df_conc_not_null.apply(lambda x: binalize_activity(x, 10), axis=1)

In [13]:
df_conc_not_null[["t_100n", "t_1u", "t_10u"]].isnull().sum()

t_100n    309878
t_1u      369273
t_10u     363424
dtype: int64

大部分はグレーゾーンにあり、白黒がつかないレコードばかり

In [14]:
# t_100n, t_1u, t_10u いずれもnullのデータを削除
df_act_not_null = df_conc_not_null.dropna(subset=["t_100n", "t_1u", "t_10u"], how='all').reset_index(drop=True)
df_act_not_null

,activity_id,assay_id,target_chembl_id,target_name,target_organism,target_accession,target_sequence,compound_chembl_id,smiles,standard_type,...,pchembl_value,description,protein_class_desc,assay_conc_full,assay_conc,assay_conc_units,assay_conc_uM,t_100n,t_1u,t_10u
0,31911,50861,CHEMBL1978,Cytochrome P450 19A1,Homo sapiens,P11511,MVLEMLNPIHYNITSIVPEAMPAATMPVLLLTGLFLLVWNYEGTSS...,CHEMBL132530,C[C@]12CCC(=O)C(O)=C1CC[C@@H]1[C@@H]2CC[C@]2(C...,Inhibition,...,None,"Inhibition of 0.25 uM [1-3H]-androst-4-ene-3,1...",enzyme cytochrome p450 cyp_19 cyp_19a cyp_...,0.25 uM,0.25,uM,0.25,NaN,1.0,1.0
1,31912,50862,CHEMBL1978,Cytochrome P450 19A1,Homo sapiens,P11511,MVLEMLNPIHYNITSIVPEAMPAATMPVLLLTGLFLLVWNYEGTSS...,CHEMBL132530,C[C@]12CCC(=O)C(O)=C1CC[C@@H]1[C@@H]2CC[C@]2(C...,Inhibition,...,None,"Inhibition of 1.25 uM [1-3H]-androst-4-ene-3,1...",enzyme cytochrome p450 cyp_19 cyp_19a cyp_...,1.25 uM,1.25,uM,1.25,NaN,NaN,1.0
2,32562,31621,CHEMBL3081,Aldose reductase,Bos taurus,P16116,AHNIVLYTGAKMPILGLGTWKSPPGKVTEAVKVAIDLGYRHIDCAH...,CHEMBL55842,COc1ccc2c(=O)c(Cc3ccc(O)cc3)coc2c1,Inhibition,...,None,The compound was tested for Inhibitory effect ...,enzyme reductase,77 uM,77,uM,77.00,0.0,0.0,0.0
3,32986,199834,CHEMBL3890,Selectin E,Homo sapiens,P16581,MIASQFLSALTLVLLIKESGAWSYNTSTEAMTYDEASAYCQQRYTH...,CHEMBL2303866,O=C(O)Cc1cccc(-c2cc(CCCC(=O)N3CCN(C(=O)CCCc4cc...,Inhibition,...,None,Tested in vitro for the concentration to inhib...,adhesion,3 mM,3,mM,3000.00,0.0,0.0,0.0
4,32987,200020,CHEMBL5378,P-selectin,Homo sapiens,P16109,MANCQIAILYQRFQRVVFGISQLLCFSALISELTNQKEVAAWTYHY...,CHEMBL2303866,O=C(O)Cc1cccc(-c2cc(CCCC(=O)N3CCN(C(=O)CCCc4cc...,Inhibition,...,None,Tested in vitro for the concentration to inhib...,adhesion,3 mM,3,mM,3000.00,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
397286,25111824,2304392,CHEMBL5291573,HTH-type transcriptional regulator LuxR,Vibrio harveyi,P21308,MDSIAKRPRTRLSPLKRKQQLMEIALEVFARRGIGRGGHADIAEIA...,CHEMBL2299418,CN(C)c1ccc(/C=C/C=O)cc1,Inhibition,...,None,Inhibition of LuxR quorum sensing system in Vi...,unclassified,100 uM,100,uM,100.00,0.0,0.0,0.0
397287,25111825,2304392,CHEMBL5291573,HTH-type transcriptional regulator LuxR,Vibrio harveyi,P21308,MDSIAKRPRTRLSPLKRKQQLMEIALEVFARRGIGRGGHADIAEIA...,CHEMBL5289639,O=C/C=C/c1ccc([N+](=O)[O-])cc1,Inhibition,...,None,Inhibition of LuxR quorum sensing system in Vi...,unclassified,100 uM,100,uM,100.00,0.0,0.0,0.0
397288,25112624,2304634,CHEMBL4599,Tyrosine kinase non-receptor protein 2,Homo sapiens,Q07912,MQPEEGTGWLLELLSEVQLQQYFLRLRDDLNVTRLSHFEYVKNEDL...,CHEMBL428690,CN1CC[C@H](c2c(O)cc(O)c3c(=O)cc(-c4ccccc4Cl)oc...,Inhibition,...,None,Inhibition of ACK1 (unknown origin) at 10 uM r...,enzyme kinase protein kinase tk ack,10 uM,10,uM,10.00,NaN,NaN,1.0
397289,25112625,2304634,CHEMBL4599,Tyrosine kinase non-receptor protein 2,Homo sapiens,Q07912,MQPEEGTGWLLELLSEVQLQQYFLRLRDDLNVTRLSHFEYVKNEDL...,CHEMBL535,CCN(CC)CCNC(=O)c1c(C)[nH]c(/C=C2\C(=O)Nc3ccc(F...,Inhibition,...,None,Inhibition of ACK1 (unknown origin) at 10 uM r...,enzyme kinase protein kinase tk ack,10 uM,10,uM,10.00,0.0,0.0,NaN


化合物の前処理。処理内容は
- smilesの正規化
- フラグメントの選択 (すなわち塩の除去)
- 重原子数``minimum_heavy_atoms`` 未満の化合物を除去。極端に小さい化合物や、フラグメント選択後に金属や塩、溶媒などが残っている化合物を除去するため

In [14]:
mol_preprocessor = MolPreprocessor(df_act_not_null["smiles"].tolist())
df_act_not_null["smiles"] = mol_preprocessor.preprocess(num=32, minimum_heavy_atoms=4)
df_act_not_null = df_act_not_null.dropna(subset=["smiles", "target_sequence"])

100%|██████████| 397291/397291 [00:19<00:00, 20487.94it/s]


In [15]:
len(df_act_not_null)

397276

### タンパク質情報の付与
``../input/uniprot-reviewed-true-2023.02.23.tsv``を用いて、タンパク質のファミリー情報を付与する

In [16]:
from protein_utils import preprocess_sequence, classify_family

In [17]:
#タンパク質配列の前処理
df_target = df_act_not_null[["target_chembl_id", "target_name", "target_organism", "target_accession", "target_sequence", "protein_class_desc"]].drop_duplicates().reset_index(drop=True)
vocab = ["A", "C", "D", "E", "F", "G", "H", "I", "K", "L", "M", 
         "N", "P", "Q", "R", "S", "T", "V", "W", "Y", "X"]
df_target = preprocess_sequence(df_target, vocab)

100%|██████████| 2998/2998 [00:00<00:00, 7735.68it/s]


In [18]:
uniprot_df = pd.read_csv(f'../input/uniprot-reviewed-true-2023.02.23.tsv', sep='\t')

# Uniprot ID, Protein families, 3D構造の情報を付与
df_target = pd.merge(df_target, uniprot_df[['Entry', 'Protein families', "3D"]], left_on='target_accession', right_on='Entry', how='left')
fam_dicts = [classify_family(family) for family in df_target['Protein families']]
fam_df = pd.concat(fam_dicts, axis=0).reset_index(drop=True)
df_target = pd.concat([df_target, fam_df], axis=1)
df_act_not_null = pd.merge(df_act_not_null, df_target.drop("Entry", axis=1))

In [19]:
df_act_not_null.columns

Index(['activity_id', 'assay_id', 'target_chembl_id', 'target_name',
       'target_organism', 'target_accession', 'target_sequence',
       'compound_chembl_id', 'smiles', 'standard_type', 'relation',
       'standard_value', 'standard_units', 'pchembl_value', 'description',
       'protein_class_desc', 'assay_conc_full', 'assay_conc',
       'assay_conc_units', 'assay_conc_uM', 't_100n', 't_1u', 't_10u',
       'Protein families', '3D', 'superfamily', 'family', 'subfamily'],
      dtype='object')

In [20]:
df_act_not_null.to_csv("./chembl_34_activity_percent_v2.csv", index=False)

In [21]:
df_act_not_null

,activity_id,assay_id,target_chembl_id,target_name,target_organism,target_accession,target_sequence,compound_chembl_id,smiles,standard_type,...,assay_conc_units,assay_conc_uM,t_100n,t_1u,t_10u,Protein families,3D,superfamily,family,subfamily
0,31911,50861,CHEMBL1978,Cytochrome P450 19A1,Homo sapiens,P11511,MVLEMLNPIHYNITSIVPEAMPAATMPVLLLTGLFLLVWNYEGTSS...,CHEMBL132530,C[C@]12CC[C@H]3[C@@H](CCC4=C(O)C(=O)CC[C@@]43C...,Inhibition,...,uM,0.25,NaN,1.0,1.0,Cytochrome P450 family,X-ray crystallography (11),NaN,Cytochrome P450 family,NaN
1,31912,50862,CHEMBL1978,Cytochrome P450 19A1,Homo sapiens,P11511,MVLEMLNPIHYNITSIVPEAMPAATMPVLLLTGLFLLVWNYEGTSS...,CHEMBL132530,C[C@]12CC[C@H]3[C@@H](CCC4=C(O)C(=O)CC[C@@]43C...,Inhibition,...,uM,1.25,NaN,NaN,1.0,Cytochrome P450 family,X-ray crystallography (11),NaN,Cytochrome P450 family,NaN
2,36784,50861,CHEMBL1978,Cytochrome P450 19A1,Homo sapiens,P11511,MVLEMLNPIHYNITSIVPEAMPAATMPVLLLTGLFLLVWNYEGTSS...,CHEMBL3349930,C[C@]12CC[C@H]3[C@@H](CCC4=CC(=O)CC[C@@]43/C=N...,Inhibition,...,uM,0.25,0.0,NaN,NaN,Cytochrome P450 family,X-ray crystallography (11),NaN,Cytochrome P450 family,NaN
3,36785,50862,CHEMBL1978,Cytochrome P450 19A1,Homo sapiens,P11511,MVLEMLNPIHYNITSIVPEAMPAATMPVLLLTGLFLLVWNYEGTSS...,CHEMBL3349930,C[C@]12CC[C@H]3[C@@H](CCC4=CC(=O)CC[C@@]43/C=N...,Inhibition,...,uM,1.25,0.0,0.0,NaN,Cytochrome P450 family,X-ray crystallography (11),NaN,Cytochrome P450 family,NaN
4,40315,50862,CHEMBL1978,Cytochrome P450 19A1,Homo sapiens,P11511,MVLEMLNPIHYNITSIVPEAMPAATMPVLLLTGLFLLVWNYEGTSS...,CHEMBL3349931,CC(=O)NC[C@]12CCC(=O)C=C1CC[C@H]1[C@@H]3CCC(=O...,Inhibition,...,uM,1.25,0.0,0.0,NaN,Cytochrome P450 family,X-ray crystallography (11),NaN,Cytochrome P450 family,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
397224,25111813,2304392,CHEMBL5291573,HTH-type transcriptional regulator LuxR,Vibrio harveyi,P21308,MDSIAKRPRTRLSPLKRKQQLMEIALEVFARRGIGRGGHADIAEIA...,CHEMBL1490851,CC(=O)CCc1ccccc1,Inhibition,...,uM,100.00,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
397225,25111822,2304392,CHEMBL5291573,HTH-type transcriptional regulator LuxR,Vibrio harveyi,P21308,MDSIAKRPRTRLSPLKRKQQLMEIALEVFARRGIGRGGHADIAEIA...,CHEMBL83159,COc1ccccc1/C=C/C=O,Inhibition,...,uM,100.00,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
397226,25111823,2304392,CHEMBL5291573,HTH-type transcriptional regulator LuxR,Vibrio harveyi,P21308,MDSIAKRPRTRLSPLKRKQQLMEIALEVFARRGIGRGGHADIAEIA...,CHEMBL452173,COc1ccc(/C=C/C=O)cc1,Inhibition,...,uM,100.00,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
397227,25111824,2304392,CHEMBL5291573,HTH-type transcriptional regulator LuxR,Vibrio harveyi,P21308,MDSIAKRPRTRLSPLKRKQQLMEIALEVFARRGIGRGGHADIAEIA...,CHEMBL2299418,CN(C)c1ccc(/C=C/C=O)cc1,Inhibition,...,uM,100.00,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
